<a href="https://colab.research.google.com/github/APP155-Jarlego/APP155-JARLEGO/blob/main/JARLEGO%2C_Samantha_Lab_05_App_Physics_155.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#Set-Up
import numpy as np

def testcreate(n, val):
  A = np.arange(val, val + n * n).reshape(n, n)
  return np.sqrt(A)

def lu_decomp(A):
  """DooLittle LU (no pivoting): A = L U, L has 1's on its diagonal."""
  n = A.shape[0]
  L = np.identity(n)
  U = A.astype(float).copy()
  for j in range(n - 1):
    for i in range(j + 1, n):
        L[i, j] = U[i, j] / U[j, j]
        U[i, :] = U[i, :] - L[i, j] * U[j, :]
        U[i, j] = 0.0
  return L, U

def forward_sub(L, b):
  y = np.zeros(len(b))
  for i in range(len(b)):
    y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i,i]
  return y

def back_sub(U, y):
  n = len(y)
  x = np.zeros(n)
  for i in range(n - 1, -1, -1):
    x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
  return x

def lu_solve(L, U, b):
  return back_sub(U, forward_sub(L, b))

def lu_inverse(A):
  n = A.shape[0]
  L, U = lu_decomp(A)
  I = np.identity(n)
  Ainv = np.zeros((n , n))
  for i in range(n):
    Ainv[:, i] = lu_solve(L, U, I[:, i])
  return Ainv

def lu_det(A):
  L, U = lu_decomp(A)
  return np.prod(np.diag(L)) * np.prod(np.diag(U))

def gauss_elim(A, b):
  n = len(b)
  A = A.astype(float).copy()
  b = b.astype(float).copy()
  for j in range(n - 1):
    for i in range(j + 1, n):
      m = A[i, j] / A[j, j]
      A[i, :] = A[i, :] - m * A[j, :]
      b[i] = b[i] - m * b[j]
  return back_sub(A, b)

#Part 1
#a.) Generating a matrix A using n = 4 and val = 21.
A1 = testcreate(4, 21)
print("A =\n", A1)

#b.) LU decompostion and verifying that L @ U = A
L1, U1 = lu_decomp(A1)
print("L =\n", L1)
print("U =\n", U1)
print("max |L@U - A| =", np.max(np.abs(L1 @ U1 - A1)))

#c.) Inverse (A x_i = e_i)
inv_lu = lu_inverse(A1)
print("Inverse (LU) =\n", inv_lu)

#d.) Determinant from det(L) and det(U)
det_lu = lu_det(A1)
print("det(L) =", np.prod(np.diag(L1)), "det (U) =", np.prod(np.diag(U1)))
print("det (A) = det(L)det(U) =", det_lu)

#e.) Comparing the np.linalg.inv() and np.linalg.det()
print("np.linalg.inv =\n", np.linalg.inv(A1))
print("max |inv_lu - inv_np =", np.max(np.abs(inv_lu - np.linalg.inv(A1))))
print("det (LU) =", det_lu)
print("det (numpy) =", np.linalg.det(A1))


#Part 2
#Set-Up
A = np.array([4., 4, 8, 4, 4, 5, 3, 7, 8,
              3, 9, 9, 4, 7, 9, 5]).reshape(4, 4)
bs = np.array([1., 2, 3, 4])
L, U = lu_decomp(A)
print("L =\n", L)
print("U =\n", U)

#a.) Inverse and Determinant utilizing code from part 1
with np.errstate(all="ignore"):
    Ainv_lu = lu_inverse(A)
    print("det(A) (LU) =", lu_det(A))
    print("Inverse (LU) =\n", Ainv_lu)

#b) Condition Number
with np.errstate(all="ignore"):
    kA = np.linalg.norm(A, 1) * np.linalg.norm(Ainv_lu, 1)
    print("k(A) using my LU inverse =", kA)
    try:
      kA_np = np.linalg.norm(A, 1) * np.linalg.norm(np.linalg.inv(A), 1)
      print("k(A) using np.linalg.inv =", kA)
    except np.linalg.LinAlgError:
      print("k(A) using np.linalg.inv  = raised LinAlgError (Singular matrix)")

#c.) Solving A x = b using Gaussian Elimination
with np.errstate(all="ignore"):
  print("x (Gaussian Elimination) = ", gauss_elim(A, bs))

#d.) Solving A x = b using LU Decomposition
with np.errstate(all="ignore"):
  print("x (LU decomposition) = ", lu_solve(L, U, bs))


#Code for Discussion Questions
#1.) What is another way in which the results for the inverse of matrix A
#can be tested other than comparing with the results of np.linalg.inv()?
print("A1 @ inv_lu =\n", np.round(A1 @ inv_lu, 6))
print("Is it the identity?", np.allclose(A1 @ inv_lu, np.identity(4), atol=1e-6))

#2.)For the matrix A in part 2, you may find that you run into an issue with
#the results. Examine the elements of the L and U matrices.
#Does one of the elements have a value that could be the cause of this issue?
#If so, what is it?
print("L =\n", L)
print("U =\n", L)
print("Diagonal of U:", np.diag(U), "-> U[3, 3] =", U[3,3])

#3.) Can the issue be solved by changing the element in question to some number?
#If so, what is this number and does it solve the issue?
y = forward_sub(L, bs)
print("from L y = b:", y, "-> last equation reads 0*x4 =", y[3])
for val in [1e-3, 1e-8, 1e-12, 1.0]:
    U_mod = U.copy()
    U_mod[3, 3] = val
    x = lu_solve(L, U_mod, bs)
    print(f"U[3, 3]={val:<7g} x={x}  |A x b|={np.linalg.norm(A @ x - bs):.2f}")

A =
 [[4.58257569 4.69041576 4.79583152 4.89897949]
 [5.         5.09901951 5.19615242 5.29150262]
 [5.38516481 5.47722558 5.56776436 5.65685425]
 [5.74456265 5.83095189 5.91607978 6.        ]]
L =
 [[1.         0.         0.         0.        ]
 [1.09108945 1.         0.         0.        ]
 [1.1751393  1.85941823 1.         0.        ]
 [1.25356634 2.61726883 2.4861338  1.        ]]
U =
 [[ 4.58257569e+00  4.69041576e+00  4.79583152e+00  4.89897949e+00]
 [ 0.00000000e+00 -1.86436436e-02 -3.65287620e-02 -5.37222161e-02]
 [ 0.00000000e+00  0.00000000e+00 -8.35038552e-05 -2.37019696e-04]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 -1.04434106e-06]]
max |L@U - A| = 0.0
Inverse (LU) =
 [[ -428980.43546836  1637250.27661743 -2014582.42462556   805708.56182695]
 [ 1354917.33545168 -5189903.386803    6403036.85326262 -2566061.84736429]
 [-1424042.66481087  5473024.88221084 -6769070.99265713  2717913.04936848]
 [  498099.53345588 -1920343.60879937  2380576.52043416  -957541.59344657]]
d